In [1]:
%pip install pillow opencv-python

  Using cached numpy-2.2.6-cp310-cp310-macosx_10_9_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp310-cp310-macosx_10_9_x86_64.whl (21.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


## Why YOLO26 Makes This Manageable

The Week-1 assessment had you train YOLO26 in PyTorch. YOLO26's **end-to-end NMS-free** design means its ONNX export is unusually clean: the model directly outputs final detections (no separate NMS post-processing required). One-line export, one straightforward inference function, and you're done with the conversion step.

```python
from ultralytics import YOLO
model = YOLO("runs/cats_v2/weights/best.pt")
model.export(format="onnx", imgsz=640, opset=17)
# -> runs/cats_v2/weights/best.onnx
```

You'll then load that `.onnx` file in `onnxruntime` (a tiny CPU-friendly runtime) and run it from your container.

## Part A — Improve the Detector (in a notebook)

Open `m6-09-assessment.ipynb` at the root of your repository.

### 1. Recap

In a markdown cell, briefly recap your Week-1 result:
- Best validation mAP@0.5 and mAP@0.5:0.95 from `m6-04-assessment`.
- Two specific weaknesses you observed in the failure cases.


In [1]:
from ultralytics import YOLO
model = YOLO("best.pt")
model.export(format="onnx", imgsz=640, opset=17)
# -> runs/cats_v2/weights/best.onnx

Ultralytics 8.4.50 🚀 Python-3.10.18 torch-2.2.2 CPU (Intel Core i7-7700HQ 2.80GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.21.0 opset 17...


/opt/miniconda3/envs/torch_env/lib/python3.10/site-packages/torch/onnx/symbolic_opset9.py:5859: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 4.2s, saved as 'best.onnx' (9.4 MB)

Export complete (4.8s)
Results saved to /Users/orhan/Desktop/ITSkillSprint/m6-09-assessment/best.onnx
Predict:         yolo predict task=detect model=best.onnx imgsz=640 
Validate:        yolo val task=detect model=best.onnx imgsz=640 data=/content/DATASET/data.yaml  
Visualize:       https://netron.app


'best.onnx'

In [4]:
import numpy as np
import torch
import onnxruntime as ort
import cv2

# 1. Define model paths and a sample test image
PT_MODEL_PATH = "best.pt"
ONNX_MODEL_PATH = "best.onnx"
IMAGE_PATH = "/Users/orhan/Desktop/ITSkillSprint/m6-09-assessment/images/0a0df46ca3f886c9(1).jpg"

# 2. Load and preprocess the image
# YOLO models expect a 640x640 image, converted from BGR to RGB, and normalized to [0, 1]
img = cv2.imread(IMAGE_PATH)
img_resized = cv2.resize(img, (640, 640))
img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

# Convert HWC to CHW format, change data type to float32, and add the batch dimension [1, C, H, W]
input_tensor = img_rgb.transpose(2, 0, 1).astype(np.float32) / 255.0
input_tensor = np.expand_dims(input_tensor, axis=0)

print("--- Running inference with PyTorch Model ---")
pt_model = YOLO(PT_MODEL_PATH)

# Extract raw tensor outputs directly from the underlying PyTorch model to compare with ONNX
with torch.no_grad():
    torch_tensor = torch.from_numpy(input_tensor)
    pt_output = pt_model.model(torch_tensor)[0].cpu().numpy()

print("--- Running inference with ONNX Runtime ---")
# Initialize the ONNX runtime inference session
ort_session = ort.InferenceSession(ONNX_MODEL_PATH)
input_name = ort_session.get_inputs()[0].name

# Run the ONNX model using the same input tensor
onnx_output = ort_session.run(None, {input_name: input_tensor})[0]

print("\n--- Comparing Model Outputs ---")
print(f"PyTorch output shape: {pt_output.shape}")
print(f"ONNX output shape:    {onnx_output.shape}")

# Filter out low-confidence boxes (e.g., score > 0.25) to compare only real detections
# Index 4 corresponds to the confidence score in [x1, y1, x2, y2, score, class_id]
CONF_THRESHOLD = 0.25

pt_mask = pt_output[0, :, 4] > CONF_THRESHOLD
onnx_mask = onnx_output[0, :, 4] > CONF_THRESHOLD

filtered_pt = pt_output[0, pt_mask]
filtered_onnx = onnx_output[0, onnx_mask]

print(f"Filtered PyTorch detections count: {len(filtered_pt)}")
print(f"Filtered ONNX detections count:    {len(filtered_onnx)}")

# Verify that both outputs match within a small numerical tolerance for actual detections
try:
    if len(filtered_pt) == 0 and len(filtered_onnx) == 0:
        print("✅ Sanity Check PASSED! Neither model detected any objects with high confidence.")
    else:
        np.testing.assert_allclose(filtered_pt, filtered_onnx, rtol=1e-02, atol=1e-02)
        print("✅ Sanity Check PASSED! The actual detections match perfectly.")
except AssertionError as e:
    print("❌ Sanity Check FAILED! There is a discrepancy between real detections.")
    print(e)

--- Running inference with PyTorch Model ---
--- Running inference with ONNX Runtime ---

--- Comparing Model Outputs ---
PyTorch output shape: (1, 300, 6)
ONNX output shape:    (1, 300, 6)
Filtered PyTorch detections count: 1
Filtered ONNX detections count:    1
✅ Sanity Check PASSED! The actual detections match perfectly.


### 2. Pick **at least three** Week-2 techniques

Apply at least three of the following — the bigger and more diverse, the better:

- **Different YOLO26 variant** — re-fine-tune from a different size (`yolo26n` / `s` / `m` / `l` / `x`) than the one you picked in Week 1. Going *up* a size (e.g. `n → s` or `s → m`) is the obvious move if your Week-1 run looked under-fit; going *down* a size with stronger augmentation can also be a smart trade if you want a smaller image to ship.
- **Stronger augmentation** — turn on / tune Ultralytics' `mosaic`, `mixup`, `copy_paste`, `hsv_h`, `hsv_s`, `hsv_v`, `degrees`, `translate`, `scale`, `flipud`, `fliplr`.
- **Longer training + cosine schedule** — increase epochs (e.g. 60–100) and use the cosine LR schedule (`cos_lr=True`).
- **Two-stage transfer learning** — train the head only for a few epochs, then unfreeze the backbone with a smaller LR.
- **Better regularisation** — add `weight_decay`, tune `dropout` if your variant supports it, use early stopping (`patience=...`).
- **More data discipline** — fix mislabelled / low-quality images you spotted in Week 1; balance class counts if uneven.

For each technique you apply, the notebook must contain:
- A short markdown explanation of *why* you're trying it.
- A code cell with the actual training run.
- The training/validation curves and final test-set metrics.

### 3. Compare against your Week-1 baseline

Produce a comparison table:

| Run | Backbone | Tricks | mAP@0.5 | mAP@0.5:0.95 | P | R |
|---|---|---|---|---|---|---|
| Week-1 baseline | yolo26&lt;your variant&gt; | none | … | … | … | … |
| v2 — run 1 | … | … | … | … | … | … |
| v2 — run 2 | … | … | … | … | … | … |
| **v2 — best** | … | … | … | … | … | … |

Choose your **best** run as the model you'll ship.

### 4. Export to ONNX

```python
from ultralytics import YOLO
model = YOLO("runs/<your-best-run>/weights/best.pt")
model.export(format="onnx", imgsz=640, opset=17, dynamic=False)
```

Sanity-check the export by:
- Loading `best.onnx` with `onnxruntime` and running inference on a handful of test images.
- Confirming that the boxes from the ONNX model match the boxes from the original PyTorch model (within tiny numerical tolerance).


## Part B — Containerise the Inference

Now you'll build a Docker image that wraps your ONNX model with a fixed CLI that the instructor can run on **unseen images**. Every student's container will follow exactly the same interface so the leaderboard run is fully automated.

### B.1 Repository layout

Inside your repo, create a `container/` directory that looks like:

```
container/
  Dockerfile
  STUDENT.json
  requirements.txt
  app/
    __init__.py
    cli.py
    detector.py
  models/
    best.onnx
```

### B.2 STUDENT.json (required)

This single file is how the instructor identifies you on the leaderboard. Place it at `container/STUDENT.json` and **inside the image at `/app/STUDENT.json`** (the Dockerfile will copy it there). Schema — exact field names, all required:

```json
{
  "first_name": "Alice",
  "last_name": "Garcia",
  "team": "alice-garcia",
  "model": {
    "framework": "yolo26",
    "variant": "yolo26s",
    "imgsz": 640,
    "epochs_total": 80,
    "tricks": ["mosaic", "cos_lr", "two_stage_finetune"]
  },
  "notes": "anything you want — short!"
}
```

`first_name` and `last_name` are mandatory and used to populate the leaderboard. `team` is a single lowercase-with-dashes slug used as the row key (use `firstname-lastname` if you have no team name).


### B.3 The standardised CLI

Your image **must** support exactly two subcommands. The instructor will run them with `docker run`. The container must also have **`python /app/cli.py`** (or the equivalent entrypoint script) defined.

#### `info`

Prints `STUDENT.json` to stdout. Used by the leaderboard runner to register your entry.

```bash
docker run --rm <your-image> info
```

Expected output: the contents of `/app/STUDENT.json`, valid JSON, on stdout, exit code 0.

#### `predict`

Runs your ONNX model on a folder of images and writes a CSV of bounding-box predictions to a fixed path.

```bash
docker run --rm \
  -v /absolute/path/to/holdout:/data/input:ro \
  -v /absolute/path/to/results:/data/output \
  <your-image> predict
```

- **Input** — `/data/input/` will contain image files (`.jpg`, `.jpeg`, `.png`). Filenames are arbitrary; treat the path **relative to `/data/input/`** as the image identifier (subdirectories are possible — preserve the relative path).
- **Output** — write **exactly** `/data/output/predictions.csv` with the schema below. Overwrite if it already exists. Exit code 0 on success.

#### Output CSV schema

`/data/output/predictions.csv` — UTF-8, comma-separated, **with header**:

```
image_path,xmin,ymin,xmax,ymax,confidence,class
```

- `image_path` — path **relative to `/data/input/`** (e.g. `img_017.jpg` or `subdir/img_017.jpg`). Use forward slashes.
- `xmin, ymin, xmax, ymax` — **absolute pixel coordinates** of the bounding-box corners in the **original image** (top-left origin = `(0, 0)`). Floats are fine; clip to image bounds.
- `confidence` — float in `[0, 1]`, the detection score.
- `class` — class **name** as a string, matching the names in your `data.yaml` (e.g. `cat`).
- One row per detected box. **Multiple boxes per image** are expected; just write multiple rows with the same `image_path`.
- For images with **no detections**, write a single row with the `image_path` filled in and the other six fields empty:

  ```csv
  empty_img.jpg,,,,,,
  ```

Example:

```csv
image_path,xmin,ymin,xmax,ymax,confidence,class
img_001.jpg,123.4,55.0,478.2,401.7,0.91,cat
img_001.jpg,512.0,200.5,640.0,330.1,0.74,cat
img_002.jpg,,,,,,
subdir/img_003.jpg,40.1,12.0,300.5,250.0,0.88,cat
```

Stick to this format exactly. The leaderboard scoring script depends on it.

### B.4 Dockerfile

Use a small Python base image. Keep the image lean — install `onnxruntime` (CPU build), not the full `ultralytics` package, in the runtime image. Reference (your image is welcome to differ; this is a known-good starting point):

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY container/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

COPY container/app /app/app
COPY container/models /app/models
COPY container/STUDENT.json /app/STUDENT.json

ENTRYPOINT ["python", "/app/app/cli.py"]
```

`requirements.txt` minimal:

```
onnxruntime==1.18.0
numpy
pillow
opencv-python-headless
```

(Pin versions you actually tested with.)

### B.5 Inference logic

Inside `app/detector.py`, implement a class that loads `models/best.onnx` once and exposes a `predict(image_path) -> list[dict]` method returning the raw boxes (with original-image-pixel coordinates). Inside `app/cli.py`, parse the subcommand (`info` / `predict`), iterate over `/data/input/`, and write the CSV described above.

Sketch:

```python
# app/detector.py
import numpy as np, onnxruntime as ort
from PIL import Image

class CatDetector:
    def __init__(self, onnx_path, imgsz=640, conf=0.25, class_names=("cat",)):
        self.session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        self.imgsz = imgsz
        self.conf = conf
        self.class_names = class_names
        self.input_name = self.session.get_inputs()[0].name

    def predict(self, image_path: str) -> list[dict]:
        img = Image.open(image_path).convert("RGB")
        orig_w, orig_h = img.size

        # letterbox to (imgsz, imgsz), preserve aspect ratio with padding,
        # remember scale + pad so we can map predictions back to original pixels
        x, scale, (pad_x, pad_y) = self._letterbox(img, self.imgsz)
        x = (np.array(x, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None, ...]

        out = self.session.run(None, {self.input_name: x})[0]  # YOLO26 e2e: (1, 300, 6)
        out = out[0]  # (300, 6) -> [x1, y1, x2, y2, score, class]

        results = []
        for x1, y1, x2, y2, score, cls in out:
            if score < self.conf:
                continue
            # undo letterbox (input-space pixels -> original-image pixels)
            x1 = (x1 - pad_x) / scale
            y1 = (y1 - pad_y) / scale
            x2 = (x2 - pad_x) / scale
            y2 = (y2 - pad_y) / scale
            # clip to image bounds
            x1 = max(0.0, min(orig_w, x1))
            y1 = max(0.0, min(orig_h, y1))
            x2 = max(0.0, min(orig_w, x2))
            y2 = max(0.0, min(orig_h, y2))
            results.append({
                "xmin": float(x1), "ymin": float(y1),
                "xmax": float(x2), "ymax": float(y2),
                "confidence": float(score),
                "class": self.class_names[int(cls)],
            })
        return results
```

> The exact YOLO26 ONNX output shape is `(N, 300, 6)` for the default end-to-end head — six values per detection: `[x1, y1, x2, y2, score, class]`, max 300 detections per image. Confirm the shape with `session.get_outputs()[0].shape` before assuming it. If you exported with `end2end=False` you'll get the legacy `(N, nc + 4, 8400)` shape and will need to add NMS yourself — strongly recommended to stick with the default end-to-end export.

### B.6 Build, test, and publish

1. Build:

   ```bash
   docker build -t <dockerhub-username>/cat-detector:final -f container/Dockerfile .
   ```

2. Test locally — create a small folder of test images and confirm both subcommands work:

   ```bash
   docker run --rm <dockerhub-username>/cat-detector:final info

   mkdir -p /tmp/inp /tmp/out
   cp some/test/images/*.jpg /tmp/inp/
   docker run --rm \
     -v /tmp/inp:/data/input:ro \
     -v /tmp/out:/data/output \
     <dockerhub-username>/cat-detector:final predict

   cat /tmp/out/predictions.csv | head
   ```

3. Push:

   ```bash
   docker login
   docker push <dockerhub-username>/cat-detector:final
   ```

4. Verify the image works on a clean machine by pulling and running it as if you were the instructor.